# Deep Dive: CUDA — Euclidean Norm (C vs CUDA Unified Memory variants)

**Group 3:** Matthew Fabregas, Kane Ibanez, Bowen Li, John Teoxon

**Specification (from our SIMD project):** compute the Euclidean norm of a double-precision vector `A` of length `n`

$$Z = \sqrt{\sum_{i=0}^{n-1} A_i^2}$$

**Data initialization:** `A[i] = sin(i * 0.001)` (double precision), **n = 2^24 = 16,777,216 elements** (128 MB).

**Versions (one cell each):**
1. C program (sequential baseline + correctness answer key)
2. CUDA — grid-stride loop, Unified Memory, no prefetch, no page creation, no mem advise
3. CUDA — grid-stride loop + prefetch (no page creation, no mem advise)
4. CUDA — grid-stride loop + prefetch + page creation (no mem advise)
5. CUDA — grid-stride loop + prefetch + page creation + mem advise

Each kernel is executed **30 times** for block sizes **256, 512 and 1024**. Kernel time is measured with CUDA events (kernel only). Every CUDA result is checked against the C answer key.

**Grid size:** the main runs launch only as many blocks as the GPU can hold at once (number of SMs × resident blocks per SM), so each thread really loops through many elements in the grid-stride loop. A separate cell compares this with the "maximum blocks" launch (`n / blockSize`, one element per thread).

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Run the whole notebook in one go so every table comes from the same session.

In [ ]:
!nvidia-smi
!nvcc --version
!rm -f results.csv

## Version 1 — C program (baseline and sanity-check answer key)

Same algorithm as `norm_c` in our SIMD project: one sequential loop that adds `A[i] * A[i]`, then `sqrt`. The vector is initialized with `A[i] = sin(i * 0.001)`. One untimed warm-up call, then **30 timed calls** (only the function call is timed, using `clock_gettime`). The resulting `Z` is the answer key for all CUDA versions.

In [ ]:
%%writefile euclid_c.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define RUNS 30

double norm_c(const double* A, long long n) {
    double sum = 0.0;
    for (long long i = 0; i < n; i++)
        sum += A[i] * A[i];
    return sqrt(sum);
}

static double now_ms(void) {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1000.0 + t.tv_nsec / 1e6;
}

int main(int argc, char** argv) {
    int p = (argc > 1) ? atoi(argv[1]) : 24;
    long long n = 1LL << p;
    double* A = (double*)malloc(n * sizeof(double));
    for (long long i = 0; i < n; i++)
        A[i] = sin(i * 0.001);

    double z = norm_c(A, n); // warm-up (untimed)
    double total = 0, mn = 1e300, mx = 0, first = 0;
    for (int r = 0; r < RUNS; r++) {
        double t1 = now_ms();
        z = norm_c(A, n);
        double ms = now_ms() - t1;
        if (r == 0) first = ms;
        total += ms;
        if (ms < mn) mn = ms;
        if (ms > mx) mx = ms;
    }
    double avg = total / RUNS;

    printf("VERSION 1: C program (baseline / answer key)\n");
    printf("n = 2^%d = %lld elements, A[i] = sin(i*0.001)\n", p, n);
    printf("Z = %.10f\n", z);
    printf("Execution time over %d runs: avg = %.4f ms | min = %.4f ms | max = %.4f ms\n", RUNS, avg, mn, mx);

    FILE* f = fopen("results.csv", "a");
    if (f) {
        fprintf(f, "C,-,-,-,%d,%.6f,%.6f,%.6f,%.6f,%.12f,%.12f,PASS\n", p, first, avg, mn, mx, z, z);
        fclose(f);
    }
    free(A);
    return 0;
}

In [ ]:
!gcc -O2 -o euclid_c euclid_c.c -lm
!./euclid_c 24

## Shared CUDA code (kernel + helpers used by all four CUDA versions)

All four CUDA versions run the **same kernel**, so the only difference between them is how the Unified Memory is handled.

**Kernel (`norm_kernel`)** — a reduction, because every element adds into one sum:
1. **Grid-stride loop:** each thread squares and adds its elements `index, index + stride, index + 2*stride, ...` into a private register `local`, where `stride` = total number of threads in the grid.
2. **Shared memory:** each block adds its threads' `local` values together with a tree reduction (`__syncthreads()` between steps). The shared array has one `double` per thread (`blockSize * 8` bytes, given at launch).
3. **atomicAdd:** thread 0 of each block adds the block's total into the global `sum`. Without the atomic, blocks would overwrite each other (race condition, see lecture 4). Only one atomic per block is needed instead of one per element.

The host takes `sqrt(sum)` after the kernel.

**Grid size (`gridMode`):**
- `gridMode 1` (**used for all main runs**): `numBlocks = number of SMs × resident blocks per SM` (found with `cudaOccupancyMaxActiveBlocksPerMultiprocessor`). This is the number of blocks the GPU can run at the same time, so the grid is much smaller than `n` and **every thread loops** through about `n / (numBlocks × blockSize)` elements. On a T4 (40 SMs, max 1024 threads per SM) this gives 160 / 80 / 40 blocks for block size 256 / 512 / 1024, about 410 elements per thread.
- `gridMode 0` (comparison only): `numBlocks = (n + blockSize - 1) / blockSize`, the maximum number of blocks, one element per thread. With this launch `stride ≥ n`, so the grid-stride loop runs only once per thread and there are 65,536 blocks (and atomics) at block size 256.

**Correctness check:** the reference `Z` is computed with the same C function as Version 1. Because the GPU adds the numbers in a different order, results can differ in the last digits, so a result is `CORRECT` if its relative difference from C is at most `1e-9` (same tolerance as our SIMD project).

**Timing:** for each of the 30 runs, `sum` is reset with `cudaMemset` (not timed), then only the kernel is timed with CUDA events. We report run 1 separately because it includes any page migration.

**CUDA 12 vs 13:** CUDA 13 changed `cudaMemPrefetchAsync` and `cudaMemAdvise` to take a `cudaMemLocation`. The small wrappers below pick the correct call for whatever toolkit Colab has.

**Program arguments:** `./program blockSize [gridMode] [power]` (defaults: 256, 1, 24).

In [ ]:
%%writefile common.cuh
#pragma once
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define RUNS 30

#define CUDA_CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
    printf("CUDA error %s at line %d: %s\n", #x, __LINE__, cudaGetErrorString(e)); exit(1); } } while (0)

// ---------------- kernels ----------------

// Sum of squares with a grid-stride loop, shared-memory block reduction, one atomicAdd per block.
// Shared memory size (blockDim.x doubles) is given as the 3rd launch parameter.
__global__ void norm_kernel(const double* A, long long n, double* sum) {
    extern __shared__ double s[];
    long long index  = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;

    double local = 0.0;
    for (long long i = index; i < n; i += stride)
        local += A[i] * A[i];

    s[threadIdx.x] = local;
    __syncthreads();
    for (int k = blockDim.x / 2; k > 0; k >>= 1) {   // blockDim is 256/512/1024 (power of 2)
        if (threadIdx.x < k)
            s[threadIdx.x] += s[threadIdx.x + k];
        __syncthreads();
    }
    if (threadIdx.x == 0)
        atomicAdd(sum, s[0]);
}

// Initializes A on the GPU (used by the "page creation" versions), grid-stride loop.
__global__ void init_kernel(double* A, long long n) {
    long long index  = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;
    for (long long i = index; i < n; i += stride)
        A[i] = sin(i * 0.001);
}

// ---------------- host helpers ----------------

// Same initialization as Version 1 (CPU)
inline void init_host(double* A, long long n) {
    for (long long i = 0; i < n; i++)
        A[i] = sin(i * 0.001);
}

// C answer key: same values and same order as norm_c in Version 1 (not timed)
inline double reference_norm(long long n) {
    double sum = 0.0;
    for (long long i = 0; i < n; i++) {
        double a = sin(i * 0.001);
        sum += a * a;
    }
    return sqrt(sum);
}

// gridMode 0: one element per thread (maximum number of blocks, the loop runs once)
// gridMode 1: blocks that fit on the GPU at once (SMs x resident blocks per SM), every thread loops
inline int pick_grid(long long n, int blockSize, int gridMode) {
    long long oneEach = (n + blockSize - 1) / blockSize;
    if (gridMode == 0) return (int)oneEach;
    int dev = 0, sms = 0, perSM = 0;
    CUDA_CHECK(cudaGetDevice(&dev));
    CUDA_CHECK(cudaDeviceGetAttribute(&sms, cudaDevAttrMultiProcessorCount, dev));
    CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&perSM, norm_kernel, blockSize,
                                                             blockSize * sizeof(double)));
    long long g = (long long)sms * perSM;
    return (int)(g < oneEach ? g : oneEach);
}

inline void prefetch_to_gpu(const void* p, size_t bytes, int dev) {
#if CUDART_VERSION >= 13000
    cudaMemLocation loc = {};
    loc.type = cudaMemLocationTypeDevice;
    loc.id = dev;
    CUDA_CHECK(cudaMemPrefetchAsync(p, bytes, loc, 0, 0));
#else
    CUDA_CHECK(cudaMemPrefetchAsync(p, bytes, dev, 0));
#endif
}

inline void advise_gpu(const void* p, size_t bytes, cudaMemoryAdvise advice, int dev) {
#if CUDART_VERSION >= 13000
    cudaMemLocation loc = {};
    loc.type = cudaMemLocationTypeDevice;
    loc.id = dev;
    CUDA_CHECK(cudaMemAdvise(p, bytes, advice, loc));
#else
    CUDA_CHECK(cudaMemAdvise(p, bytes, advice, dev));
#endif
}

// Runs the kernel RUNS times, times each run, checks correctness, prints and appends to results.csv
inline void run_and_report(const char* name, const char* label, const double* A, long long n,
                           int p, int blockSize, int gridMode, double* sum) {
    int numBlocks = pick_grid(n, blockSize, gridMode);
    size_t shmem = blockSize * sizeof(double);
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    double times[RUNS];
    for (int r = 0; r < RUNS; r++) {
        CUDA_CHECK(cudaMemset(sum, 0, sizeof(double)));   // reset sum (not timed)
        CUDA_CHECK(cudaEventRecord(start));
        norm_kernel<<<numBlocks, blockSize, shmem>>>(A, n, sum);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        CUDA_CHECK(cudaGetLastError());
        float ms = 0;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        times[r] = ms;
    }
    CUDA_CHECK(cudaDeviceSynchronize());
    double z = sqrt(*sum);                                   // result of the last run

    double total = 0, mn = 1e300, mx = 0, warm = 0;
    for (int r = 0; r < RUNS; r++) {
        total += times[r];
        if (times[r] < mn) mn = times[r];
        if (times[r] > mx) mx = times[r];
        if (r > 0) warm += times[r];
    }
    double avg = total / RUNS;
    warm /= (RUNS - 1);

    double ref = reference_norm(n);
    double rel = fabs(z - ref) / fabs(ref);
    bool ok = rel <= 1e-9;
    double perThread = (double)n / ((double)numBlocks * blockSize);

    printf("%s\n", name);
    printf("n = 2^%d = %lld | block size = %d | blocks = %d (gridMode %d, %s) | elements per thread = %.1f\n",
           p, n, blockSize, numBlocks, gridMode, gridMode == 0 ? "one element per thread" : "blocks that fit on the GPU",
           perThread);
    printf("Kernel time over %d runs: avg = %.4f ms | min = %.4f ms | max = %.4f ms\n", RUNS, avg, mn, mx);
    printf("Run 1 = %.4f ms | avg of runs 2-%d = %.4f ms | bandwidth (avg 2-%d) = %.1f GB/s\n",
           times[0], RUNS, warm, RUNS, (double)n * 8.0 / (warm * 1e6));
    printf("Z (CUDA) = %.10f | Z (C answer key) = %.10f | rel diff = %.2e -> %s\n\n",
           z, ref, rel, ok ? "CORRECT" : "WRONG");

    FILE* f = fopen("results.csv", "a");
    if (f) {
        fprintf(f, "%s,%d,%d,%d,%d,%.6f,%.6f,%.6f,%.6f,%.12f,%.12f,%s\n",
                label, blockSize, gridMode, numBlocks, p, times[0], avg, mn, mx, z, ref, ok ? "PASS" : "FAIL");
        fclose(f);
    }
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
}

// args: blockSize [gridMode, default 1] [power p, default 24]
inline void parse_args(int argc, char** argv, int* blockSize, int* gridMode, int* p) {
    *blockSize = (argc > 1) ? atoi(argv[1]) : 256;
    *gridMode  = (argc > 2) ? atoi(argv[2]) : 1;
    *p         = (argc > 3) ? atoi(argv[3]) : 24;
    if (*blockSize < 32 || *blockSize > 1024 || (*blockSize & (*blockSize - 1)) != 0) {
        printf("blockSize must be a power of two between 32 and 1024\n");
        exit(1);
    }
}

## Version 2 — CUDA Unified Memory (grid-stride loop, no prefetch, no page creation, no mem advise)

`A` is allocated with `cudaMallocManaged` and initialized on the **CPU**, so its pages are created in CPU memory. Nothing else is done: when the kernel first touches `A`, the GPU page-faults and the pages migrate to the GPU on demand. This migration cost lands inside **run 1** of the kernel.

In [ ]:
%%writefile cuda_unified.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, gridMode, p;
    parse_args(argc, argv, &blockSize, &gridMode, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    init_host(A, n);   // pages created on the CPU; no prefetch, no advise

    run_and_report("VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)",
                   "CUDA Unified", A, n, p, blockSize, gridMode, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

In [ ]:
!nvcc -O2 -arch=sm_75 -o cuda_unified cuda_unified.cu
!for b in 256 512 1024; do ./cuda_unified $b 1 24; done

## Version 3 — CUDA Unified Memory + prefetch (no page creation, no mem advise)

Same as Version 2 (`A` initialized on the **CPU**), but before the kernel we call `cudaMemPrefetchAsync(A, bytes, device)`. The whole vector is copied to the GPU in large chunks **before** the kernel starts, so the kernel itself does not page-fault.

In [ ]:
%%writefile cuda_prefetch.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, gridMode, p;
    parse_args(argc, argv, &blockSize, &gridMode, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);
    int device = -1;
    CUDA_CHECK(cudaGetDevice(&device));

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    init_host(A, n);                        // pages created on the CPU
    prefetch_to_gpu(A, bytes, device);      // migrate A to the GPU before the kernel
    prefetch_to_gpu(sum, sizeof(double), device);
    CUDA_CHECK(cudaDeviceSynchronize());

    run_and_report("VERSION 3: CUDA Prefetch (no page creation, no mem advise)",
                   "CUDA Prefetch", A, n, p, blockSize, gridMode, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

In [ ]:
!nvcc -O2 -arch=sm_75 -o cuda_prefetch cuda_prefetch.cu
!for b in 256 512 1024; do ./cuda_prefetch $b 1 24; done

## Version 4 — CUDA Unified Memory + prefetch + page creation (no mem advise)

**Our definition of page creation:** `cudaMallocManaged` only reserves addresses; the physical pages are created when the memory is first touched. Here we create the pages **directly in GPU memory**: we prefetch the still-empty `A` to the GPU, then initialize it with a GPU kernel (`init_kernel`, same formula `sin(i * 0.001)`, also a grid-stride loop). The data is never written by the CPU, so nothing has to be migrated from CPU to GPU at all.

In [ ]:
%%writefile cuda_pages.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, gridMode, p;
    parse_args(argc, argv, &blockSize, &gridMode, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);
    int device = -1;
    CUDA_CHECK(cudaGetDevice(&device));

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    prefetch_to_gpu(A, bytes, device);      // prefetch the empty buffer to the GPU
    prefetch_to_gpu(sum, sizeof(double), device);
    init_kernel<<<pick_grid(n, blockSize, 1), blockSize>>>(A, n);   // page creation on the GPU
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    run_and_report("VERSION 4: CUDA Prefetch + page creation (no mem advise)",
                   "CUDA Prefetch+Page creation", A, n, p, blockSize, gridMode, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

In [ ]:
!nvcc -O2 -arch=sm_75 -o cuda_pages cuda_pages.cu
!for b in 256 512 1024; do ./cuda_pages $b 1 24; done

## Version 5 — CUDA Unified Memory + prefetch + page creation + mem advise

Same as Version 4, plus `cudaMemAdvise` hints (lecture 2):
- `cudaMemAdviseSetPreferredLocation` = GPU for `A` and `sum`, set **before** the prefetch, so the driver keeps these pages in GPU memory.
- `cudaMemAdviseSetReadMostly` for `A`, set **after** it is initialized, because the norm kernel only reads `A` (it is a source operand).

In [ ]:
%%writefile cuda_advise.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, gridMode, p;
    parse_args(argc, argv, &blockSize, &gridMode, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);
    int device = -1;
    CUDA_CHECK(cudaGetDevice(&device));

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    advise_gpu(A, bytes, cudaMemAdviseSetPreferredLocation, device);           // mem advise
    advise_gpu(sum, sizeof(double), cudaMemAdviseSetPreferredLocation, device);
    prefetch_to_gpu(A, bytes, device);                                         // prefetch
    prefetch_to_gpu(sum, sizeof(double), device);
    init_kernel<<<pick_grid(n, blockSize, 1), blockSize>>>(A, n);              // page creation on the GPU
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    advise_gpu(A, bytes, cudaMemAdviseSetReadMostly, device);                  // A is read-only from now on
    CUDA_CHECK(cudaDeviceSynchronize());

    run_and_report("VERSION 5: CUDA Prefetch + page creation + mem advise",
                   "CUDA Prefetch+Page creation+memadvise", A, n, p, blockSize, gridMode, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

In [ ]:
!nvcc -O2 -arch=sm_75 -o cuda_advise cuda_advise.cu
!for b in 256 512 1024; do ./cuda_advise $b 1 24; done

## Block size and number of blocks: "blocks that fit on the GPU" vs "maximum blocks"

Guide question (a) asks how block size affects execution time when using the maximum number of blocks. This cell runs all four versions again with **`gridMode 0`**: `numBlocks = (n + blockSize - 1) / blockSize`, one element per thread (65,536 / 32,768 / 16,384 blocks). Compare the times with the main runs above (`gridMode 1`, 160 / 80 / 40 blocks on a T4).

With `gridMode 0` the grid-stride loop runs only once per thread, and every one of the many blocks does its own shared-memory reduction and `atomicAdd`.

In [ ]:
%%bash
for v in cuda_unified cuda_prefetch cuda_pages cuda_advise; do
  for b in 256 512 1024; do ./$v $b 0 24; done
done

## Other vector sizes: 2^20 and 2^26 (same sizes as our SIMD project)

Guide question (a) asks how block size affects execution time while observing various numbers of elements. This cell repeats everything at **n = 2^20** (1,048,576 elements, 8 MB) and **n = 2^26** (67,108,864 elements, 512 MB): the C baseline, then all four CUDA versions at block sizes 256 / 512 / 1024, with both launches (`gridMode 1` = blocks that fit on the GPU, `gridMode 0` = maximum blocks, one element per thread). Same data initialization, 30 runs and correctness check as the main runs. The summary tables are printed in the *Results summary* section.

In [ ]:
%%bash
for p in 20 26; do
  echo "################ n = 2^$p ################"
  ./euclid_c $p
  for v in cuda_unified cuda_prefetch cuda_pages cuda_advise; do
    for g in 1 0; do
      for b in 256 512 1024; do ./$v $b $g $p; done
    done
  done
done

## Boundary test (same sizes as our SIMD project)

Checks the CUDA kernel on very small and odd sizes (n = 1, 3, 1000, 1001, 1002, 1003), where the number of elements does not divide evenly by the block size or the grid. The grid-stride loop's `i < n` check must handle the leftover elements (the SIMD versions needed a separate tail loop for this). Uses the main launch (`gridMode 1`, block size 256).

In [ ]:
%%writefile boundary.cu
#include "common.cuh"

int main() {
    long long sizes[6] = {1, 3, 1000, 1001, 1002, 1003};
    int blockSize = 256, fails = 0;
    printf("boundary test (block size %d, gridMode 1)\n", blockSize);
    for (int k = 0; k < 6; k++) {
        long long n = sizes[k];
        double *A, *sum;
        CUDA_CHECK(cudaMallocManaged(&A, n * sizeof(double)));
        CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));
        init_host(A, n);
        int grid = pick_grid(n, blockSize, 1);
        CUDA_CHECK(cudaMemset(sum, 0, sizeof(double)));
        norm_kernel<<<grid, blockSize, blockSize * sizeof(double)>>>(A, n, sum);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaDeviceSynchronize());
        double z = sqrt(*sum), ref = reference_norm(n);
        bool ok = fabs(z - ref) <= 1e-9 * fabs(ref);   // also works when ref = 0 (n = 1)
        if (!ok) fails++;
        printf("n = %4lld  blocks = %d  C = %.10f  CUDA = %.10f  %s\n", n, grid, ref, z, ok ? "correct" : "WRONG");
        CUDA_CHECK(cudaFree(A));
        CUDA_CHECK(cudaFree(sum));
    }
    printf(fails ? "%d FAILED\n" : "all correct\n", fails);
    return fails;
}

In [ ]:
!nvcc -O2 -arch=sm_75 -o boundary boundary.cu
!./boundary

## Results summary (2^24 elements)

Collected from all the runs above. Speedup = C average time / version average time (same Colab machine, same size). The first table is the main runs at 2^24 (`gridMode 1`); the second compares both launches at 2^24; the last tables show 2^20, 2^24 and 2^26. x86-64, XMM and YMM come from our SIMD project (Visual Studio, Windows) and are added to the README separately.

In [ ]:
import pandas as pd
cols = ['version','block','gridMode','blocks','power','run1_ms','avg_ms','min_ms','max_ms','Z','Z_ref','correct']
df = pd.read_csv('results.csv', names=cols, dtype={'block': str, 'gridMode': str, 'blocks': str})
df = df[df['power'] == 24]
c_avg = df[df['version'] == 'C']['avg_ms'].iloc[-1]
df['speedup_vs_C'] = (c_avg / df['avg_ms']).round(2)
pd.set_option('display.width', 200)
show = ['version','block','blocks','run1_ms','avg_ms','min_ms','max_ms','speedup_vs_C','correct']

print('Main runs (gridMode 1: blocks that fit on the GPU, every thread loops)')
main = df[(df['version'] == 'C') | (df['gridMode'] == '1')]
print(main[show].round(4).to_string(index=False))

print('\nAverage kernel time (ms): gridMode 0 (n / blockSize blocks) vs gridMode 1 (blocks that fit on the GPU)')
cuda = df[df['version'] != 'C'].copy()
cuda['block'] = cuda['block'].astype(int)
pivot = cuda.pivot_table(index='version', columns=['gridMode', 'block'], values='avg_ms', aggfunc='last')
order = ['CUDA Unified', 'CUDA Prefetch', 'CUDA Prefetch+Page creation', 'CUDA Prefetch+Page creation+memadvise']
print(pivot.reindex(order).round(4).to_string())

# ---- all sizes (2^20, 2^24, 2^26): average kernel time and speedup vs C on the same size
print('\nAll sizes: average kernel time in ms (speedup vs C of the same size in brackets)')
allp = pd.read_csv('results.csv', names=cols, dtype={'block': str, 'gridMode': str, 'blocks': str})
for p in sorted(allp['power'].unique()):
    d = allp[allp['power'] == p]
    c = d[d['version'] == 'C']['avg_ms']
    if c.empty:
        continue
    c = c.iloc[-1]
    cu = d[d['version'] != 'C'].copy()
    cu['block'] = cu['block'].astype(int)
    cu['cell'] = cu['avg_ms'].map('{:.4f}'.format) + ' (' + (c / cu['avg_ms']).map('{:.1f}x'.format) + ')'
    piv = cu.pivot_table(index='version', columns=['gridMode', 'block'], values='cell', aggfunc='last')
    print(f'\nn = 2^{p} | C avg = {c:.4f} ms | all correct: {(d["correct"] == "PASS").all()}')
    print(piv.reindex(order).to_string())

## Profiling — Nsight Systems (nsys), all CUDA versions

Block size 256, main launch (`gridMode 1`). `--stats=true` prints the kernel summary, the Unified Memory transfer summary (`cuda_gpu_mem_time_sum` / `cuda_gpu_mem_size_sum`), the Unified Memory totals (`um_total_sum`, with page-fault counts) and the CUDA API summary. Take screenshots of these for the README. Expected: many page faults and small transfers in Version 2, 64 large transfers in Version 3, no Host-to-Device transfers in Versions 4 and 5. nsys slows the program down, so the times printed here are not benchmark times.

In [ ]:
!nsys profile -o nsys_cuda_unified --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_unified 256 1 24

In [ ]:
!nsys profile -o nsys_cuda_prefetch --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_prefetch 256 1 24

In [ ]:
!nsys profile -o nsys_cuda_pages --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_pages 256 1 24

In [ ]:
!nsys profile -o nsys_cuda_advise --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_advise 256 1 24

## Profiling — Nsight Compute (ncu), all CUDA versions

Profiles one `norm_kernel` launch per version (`-k norm_kernel --launch-count 1`, block size 256, `gridMode 1`). Note: the kernel times printed by the program during this cell are slowed down by the profiler (it replays the kernel many times and fixes the GPU clock), so the benchmark numbers come from the cells above, not from here.

In [ ]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_unified 256 1 24

In [ ]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_prefetch 256 1 24

In [ ]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_pages 256 1 24

In [ ]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_advise 256 1 24